<a href="https://colab.research.google.com/github/gagandeepltcs24-art/BIS_LAB/blob/main/Cuckoo_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np



weights = np.array([2, 3, 4, 5, 9, 7, 6, 8])
profits = np.array([3, 4, 5, 8, 10, 7, 9, 11])

capacity = 20
n_items = len(weights)



n = 25
Pa = 0.25
max_iter = 20
alpha = 0.01
beta = 1.5



def fitness(solution):

    total_weight = np.sum(solution * weights)
    total_profit = np.sum(solution * profits)

    if total_weight > capacity:
        return 0

    return total_profit




def levy_flight(beta):

    sigma = (
        np.math.gamma(1 + beta) *
        np.sin(np.pi * beta / 2) /
        (
            np.math.gamma((1 + beta) / 2) *
            beta * 2 ** ((beta - 1) / 2)
        )
    ) ** (1 / beta)

    u = np.random.normal(0, sigma)
    v = np.random.normal(0, 1)

    step = u / abs(v) ** (1 / beta)

    return step




def binary_conversion(position):

    sigmoid = 1 / (1 + np.exp(-np.clip(position, -50, 50)))

    return (np.random.rand(n_items) < sigmoid).astype(int)




nests = np.random.randint(0, 2, (n, n_items))

for i in range(n):

    while np.sum(nests[i] * weights) > capacity:
        selected = np.where(nests[i] == 1)[0]

        if len(selected) == 0:
            break

        nests[i, np.random.choice(selected)] = 0


fitness_values = np.array([fitness(x) for x in nests])

best_index = np.argmax(fitness_values)

best_solution = nests[best_index].copy()
best_fitness = fitness_values[best_index]



for iteration in range(max_iter):

    for i in range(n):

        step = levy_flight(beta)

        position = nests[i] + alpha * step * (
            best_solution - nests[i]
        )

        new_solution = binary_conversion(position)

        while np.sum(new_solution * weights) > capacity:

            selected = np.where(new_solution == 1)[0]

            if len(selected) == 0:
                break

            new_solution[np.random.choice(selected)] = 0

        new_fitness = fitness(new_solution)

        j = np.random.randint(0, n)

        if new_fitness > fitness_values[j]:

            nests[j] = new_solution.copy()
            fitness_values[j] = new_fitness

    abandon_count = int(Pa * n)

    worst_indices = np.argsort(fitness_values)[:abandon_count]

    for j in worst_indices:

        nests[j] = np.random.randint(0, 2, n_items)

        while np.sum(nests[j] * weights) > capacity:

            selected = np.where(nests[j] == 1)[0]

            if len(selected) == 0:
                break

            nests[j, np.random.choice(selected)] = 0

        fitness_values[j] = fitness(nests[j])

    current_best = np.argmax(fitness_values)

    if fitness_values[current_best] > best_fitness:

        best_fitness = fitness_values[current_best]
        best_solution = nests[current_best].copy()

    print(
        f"Iteration {iteration + 1}: "
        f"Best Profit = {best_fitness}"
    )




total_weight = np.sum(best_solution * weights)
total_profit = np.sum(best_solution * profits)

print("\n========== FINAL RESULT ==========")

print("Selected Items:", best_solution)

print("Selected Item Numbers:",
      np.where(best_solution == 1)[0] + 1)

print("Total Weight:", total_weight)

print("Total Profit:", total_profit)

print("Knapsack Capacity:", capacity)
